In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

import glob
import numpy as np
import torch
import torchaudio
import torch.nn as nn
from transformers import Wav2Vec2Model, Wav2Vec2Processor
from peft import PeftModel
from sklearn.metrics import roc_curve, auc
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MISC_DIR = os.path.join(BASE_DIR, "data", "misc")
os.makedirs(MISC_DIR, exist_ok=True)

# Load model
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
base_model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h")
base_model.config.mask_time_prob = 0.0
base_model.config.mask_feature_prob = 0.0
model = PeftModel.from_pretrained(base_model, os.path.join(BASE_DIR, "data", "models", "lora_concat_w2v2_960h"))
model.eval().to(DEVICE)

class ClassificationHead(nn.Module):
    def __init__(self, hidden_size=768, num_labels=2, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.linear = nn.Linear(hidden_size, num_labels)
    def forward(self, x):
        return self.linear(self.dropout(x))

head = ClassificationHead()
head.load_state_dict(torch.load(os.path.join(BASE_DIR, "data", "models", "lora_concat_head.pt"), weights_only=True))
head.eval().to(DEVICE)

def mean_pool(hidden, attn_mask):
    input_lengths = attn_mask.sum(dim=1)
    base = model.base_model.model
    output_lengths = base._get_feat_extract_output_lengths(input_lengths).long().clamp(min=1)
    out_mask = torch.arange(hidden.size(1), device=hidden.device).unsqueeze(0) < output_lengths.unsqueeze(1)
    pooled = (hidden * out_mask.unsqueeze(-1)).sum(1) / output_lengths.unsqueeze(1).float()
    return pooled

# Collect real data
REAL_DIR = os.path.join(BASE_DIR, "data", "real")
real_groups = {
    "lvPPA":     sorted(glob.glob(os.path.join(REAL_DIR, "*lvPPA*", "**", "*.wav"), recursive=True)),
    "JHU":       sorted(glob.glob(os.path.join(REAL_DIR, "*jhu*", "**", "*.wav"), recursive=True)),
    "Control":   sorted(glob.glob(os.path.join(REAL_DIR, "*segmentedcc*", "**", "*.wav"), recursive=True)),
    "Capilouto": sorted(glob.glob(os.path.join(REAL_DIR, "*Capilouto*", "**", "*.wav"), recursive=True)),
}

# Get P(dysfluent) for all clips
group_probs = {}
for name, files in real_groups.items():
    probs = []
    for f in files:
        wav, sr = torchaudio.load(f)
        wav = torchaudio.functional.resample(wav, sr, 16000).mean(0)
        inputs = processor(wav, sampling_rate=16000, return_tensors="pt")
        input_values = inputs.input_values.to(DEVICE)
        attn_mask = torch.ones_like(input_values)
        with torch.no_grad():
            hidden = model(input_values, attention_mask=attn_mask).last_hidden_state
            pooled = mean_pool(hidden, attn_mask)
            prob = torch.softmax(head(pooled), dim=1)[0, 1].item()
        probs.append(prob)
    group_probs[name] = np.array(probs)
    print(f"{name:>10s}  ({len(probs):3d} clips)  avg_P={np.mean(probs):.3f}")

# Build labels
real_true = {}
for name in real_groups:
    real_true[name] = np.ones(len(group_probs[name])) if name in ("lvPPA", "JHU") else np.zeros(len(group_probs[name]))

# Pooled controls
ctrl_true = np.zeros(len(group_probs["Control"]) + len(group_probs["Capilouto"]))
ctrl_probs = np.concatenate([group_probs["Control"], group_probs["Capilouto"]])

# All pooled
y_true_all = np.concatenate([real_true[n] for n in ["Control", "Capilouto", "lvPPA", "JHU"]])
y_prob_all = np.concatenate([group_probs[n] for n in ["Control", "Capilouto", "lvPPA", "JHU"]])
fpr_all, tpr_all, _ = roc_curve(y_true_all, y_prob_all)
auc_all = auc(fpr_all, tpr_all)

# Plot
fig, ax = plt.subplots(figsize=(6, 6))
for dys_name, color in [("lvPPA", "tab:red"), ("JHU", "tab:orange")]:
    yt = np.concatenate([ctrl_true, real_true[dys_name]])
    yp = np.concatenate([ctrl_probs, group_probs[dys_name]])
    fpr, tpr, _ = roc_curve(yt, yp)
    a = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, label=f"{dys_name} vs Controls (AUC={a:.3f})")

ax.plot(fpr_all, tpr_all, color="black", lw=2, label=f"All pooled (AUC={auc_all:.3f})")
ax.plot([0, 1], [0, 1], "k--", alpha=0.3)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve \u2014 LoRA-Concat Model on Real Data")
ax.legend(loc="lower right")
ax.set_aspect("equal")
fig.tight_layout()
fig.savefig(os.path.join(MISC_DIR, "roc_real_data_concat.png"), dpi=150)
plt.close(fig)
print(f"\nPooled AUC: {auc_all:.4f}")
print(f"Saved to {MISC_DIR}/roc_real_data_concat.png")

/data/liharrison/miniconda3/envs/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 210/210 [00:00<00:00, 815.71it/s, Materializing param=feature_projection.projection.weight]                         
Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
lm_head.bias      | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


     lvPPA  ( 89 clips)  avg_P=0.820
       JHU  ( 74 clips)  avg_P=0.924
   Control  (235 clips)  avg_P=0.257
 Capilouto  (311 clips)  avg_P=0.235

Pooled AUC: 0.9082
Saved to /data/liharrison/lvsim/data/misc/roc_real_data_concat.png
